# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [30]:
NAMEs = "Laine Goldmacher, Max Magrill, Ashley Qin"

In [13]:
%pip install pandas torch scikit-learn


  Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 284.4 kB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

- What cost function should be used for each of the below task.
- What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.
- Why it is important to standardize the data before training each model.
- How you could detect overfitting when training your models.

1. Binary Cross-Entropy (aka log loss), Categorical Cross-Entropy, and Mean Squared Error (MSE) should be used for Binary Classification, Multi-Class Classification, and Regression respectively. 
   1. Binary Cross-Entropy (BCE) measures the difference between predicted probabilities and their actual binary labels (0 or 1).
   2. Categorical Cross-Entropy measures how well the predicted probability distribution matches the true class, penalizing low probabilities assigned to the correct class.
   3. MSE measures the average squared difference between predicted and actual continuous values, penalizing larger errors more heavily.
2. For Binary Classification, there should be one output unit using a sigmoid activation function, producing a probability between 0 and 1. Predictions are classified using a threshold, typically 0.5. For Multi-Class Classification, there should also be as many output units as classes we have using a softmax activation function, producing probabilities that sum to 1. The predicted class is the one with the highest probability. Finally, for Regression, there should still be one output unit with no activation, allowing predictions to take any real value.
3. It is important to standardize the data before training each model because standardization ensures that features are on comparable scales, preventing features with larger numerical values from disproportionately influencing gradient updates. This improves the stability and convergence speed of stochastic gradient descent, making training more efficient. It should be fitted on the data before training each model to avoid data leakage; the scaler is fitted on the training data and then applied to both training and holdout sets.
4. Overfitting occurs when a model learns the patterns specific to training data rather than generalizable relationships. We can detect it by comparing training and holdout performance across epochs. If training loss continues decreasing while holdout loss increases, or if training accuracy improves while holdout accuracy stagnates or decreases, this is a sign of overfitting. A large difference in performance between the training and holdout sets is another indicator.

## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [31]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

In [32]:
# USE AS MANY CELLS AS NEEDED
#Section 1
def sigmoid(z):
    return torch.sigmoid(z)

def relu(z):
    return torch.maximum(z, torch.tensor(0.0))

class BinaryMLP(object):
    
    def __init__(self, input_dim, hidden_dim, num_layers=1, activation="relu",learning_rate=0.1):
        
        self.activation = activation
        self.learning_rate = learning_rate

        dims = [input_dim] + [hidden_dim] * num_layers + [1]

        self.weights = []
        self.biases = []
        for i in range(len(dims) - 1):
            W = torch.randn(dims[i], dims[i + 1]) * (2 / dims[i]) ** .5
            b = torch.randn(dims[i + 1])
            self.weights.append(W.requires_grad_())
            self.biases.append(b.requires_grad_())

    def activate(self, z):
        if self.activation == "relu":
            return relu(z)
        elif self.activation == "sigmoid":
            return sigmoid(z)
        elif self.activation == "tanh":
            return torch.tanh(z)

    def forward(self, X):
        a = X
        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            a = self.activate(torch.matmul(a, W) + b)
        out = torch.matmul(a, self.weights[-1]) + self.biases[-1]
        return sigmoid(out)

    def cost(self, y_pred, y):
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        return -torch.mean(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred))


    def fit(self, X, y, n_epochs=100):
        
        for epoch in range(n_epochs):

            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii], y[ii]

                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()

                with torch.no_grad():
                    for W, b in zip(self.weights, self.biases):
                        W -= self.learning_rate * W.grad
                        b -= self.learning_rate * b.grad

                    for W, b in zip(self.weights, self.biases):
                        W.grad.zero_()
                        b.grad.zero_()
            
            if epoch % 50 == 0:
                print(f"Epoch {epoch}, Cost: {cost.item():.4f}")
        return self    
        
    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X)
            predicted = (probs > 0.5).float()
        return predicted

In [33]:
# Section 2
scaler = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_scaled = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test_scaled  = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)
binaryMLP = BinaryMLP(input_dim=X_train_scaled.shape[1], hidden_dim=10, num_layers=1, learning_rate=0.1, activation="relu")
binaryMLP.fit(X_train_scaled, y_train, n_epochs=50)
y_pred = binaryMLP.predict(X_test_scaled)
print(f"Reporting set's accuracy: {accuracy_score(y_test.numpy(), y_pred.numpy()) * 100:.2f}%")

Epoch 0, Cost: 0.0024
Reporting set's accuracy: 96.50%


In [34]:
# Section 3
import pandas as pd

configs = [
    (1, 5, "relu"),
    (1, 10, "relu"),
    (2, 10, "relu"),
    (1, 5, "tanh"),
    (1, 10, "tanh"),
    (2, 10, "tanh"),
]

results = []
models = {}

for layers, units, activation in configs:
    torch.manual_seed(1234)
    model = BinaryMLP(input_dim=X_train_scaled.shape[1], hidden_dim=units, num_layers=layers, learning_rate=0.1, activation=activation)
    model.fit(X_train_scaled, y_train, n_epochs=150)
    train_acc = accuracy_score(y_train.numpy(), model.predict(X_train_scaled).numpy())
    holdout_acc = accuracy_score(y_test.numpy(), model.predict(X_test_scaled).numpy())
    # Average BCE over the full set (reshape y to (n, 1) to match forward's output)
    with torch.no_grad():
        train_bce = model.cost(model.forward(X_train_scaled), y_train.reshape(-1, 1)).item()
        holdout_bce = model.cost(model.forward(X_test_scaled), y_test.reshape(-1, 1)).item()
    results.append({
        "Layers": layers,
        "Hidden Units": units,
        "Activation": activation,
        "Train Accuracy": train_acc,
        "Holdout Accuracy": holdout_acc,
        "Train BCE": train_bce,
        "Holdout BCE": holdout_bce,
    })
    models[(layers, units, activation)] = model

results_df = pd.DataFrame(results)
results_df.round(4)

Epoch 0, Cost: 0.0863
Epoch 50, Cost: 0.0012
Epoch 100, Cost: 0.0000
Epoch 0, Cost: 1.7933
Epoch 50, Cost: 0.0029
Epoch 100, Cost: 2.3854
Epoch 0, Cost: 0.0372
Epoch 50, Cost: 0.0000
Epoch 100, Cost: 0.0067
Epoch 0, Cost: 0.0486
Epoch 50, Cost: 0.0007
Epoch 100, Cost: 0.0015
Epoch 0, Cost: 2.2301
Epoch 50, Cost: 0.0080
Epoch 100, Cost: 0.4380
Epoch 0, Cost: 0.2385
Epoch 50, Cost: 0.0000
Epoch 100, Cost: 0.0040


,Layers,Hidden Units,Activation,Train Accuracy,Holdout Accuracy,Train BCE,Holdout BCE
0,1,5,relu,0.9712,0.985,0.0849,0.0465
1,1,10,relu,0.9700,0.985,0.0772,0.0478
2,2,10,relu,0.9575,0.980,0.0887,0.0604
3,1,5,tanh,0.9738,0.975,0.0707,0.0623
4,1,10,tanh,0.9788,0.975,0.0592,0.0552
5,2,10,tanh,0.9788,0.975,0.0501,0.0606


Binary classification reflection and discussion
1. Increasing the number of hidden units from 5 to 10 almost had no effect on accuracy. For ReLU, the holdout accuracy stayed at 98.5%, and the training accuracy barely changed from 97.12% to 97.00%. For the Tanh activation function, training accuracy increased slightly from 97.38% to 97.88%, while the holdout accuracy stayed constant at 97.5%.Since accuracy only checks whether a prediction falls on the correct side of 0.5, we can also compare binary cross-entropy, which measures how confident the predictions are. For ReLU, the training BCE lowered from 0.0849 to 0.0772, but the holdout BCE stayed around 0.0465, barely increasing to 0.0478. This means that the added capacity mainly made the model more confident on the data it has already seen. For Tanh, both the training BCE (from 0.0707 to 0.0592) and the holdout BCE (from 0.0623 to 0.0552) decreased, so the extra units did produce a slightly more confident prediction on unseen data. Since the moons dataset has only two features and a single curved boundary, five hidden units already provided enough capacity, so adding additional units gave us little benefit. 
2. Adding a second hidden layer did not improve accuracy for either activation function. For ReLU, it hurt the training accuracy, where it fell from 97.00% to 95.75%. Holdout accuracy fell from 98.5% to 98.0%. Both the training BCE (from 0.0772 to 0.0887) and the holdout BCE (from 0.0478 to 0.0604) increased. Since both training and holdout results got worse, The deeper network appeared to have trained less effectively rather than overfit, likely because a deeper network is harder to optimize with stochastic gradient descent and ReLU units can become inactive ("die") during training. For Tanh, accuracy was essentially unchanged, training BCE dreased from 0.0592 to 0.05801, and holdout BCE increased from 0.0552 to 0.0606. This pattern suggests that the additional complexity may have reduced generalization, since the two-layer Tanh model achieved lower training BCE but higher holdout BCE. The deeper Tanh model remained confident on training data but became less confident on unseen data.
3. Tanh achieved a higher training accuracy and lower training BCE in every configuration, meaning it fit the training data more closely. However, ReLU generalized the data better, and it achieved a higher holdout accuracy (98.5%) for both one-layer models and a lower holdout BCE than Tanh in almost every configuration. The best overall model was the one-layer, five-unit ReLU network, which had both the highest holdout accuracy and the lowest holdout BCE (0.0465). Tanh was more stable in terms of accuracy as depth increased, whereas ReLU's accuracy dropped with the second layer. 

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [14]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [ ]:
# USE AS MANY CELLS AS NEEDED
import torch
import torch.nn.functional as F
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score

# Section 1
class MultiClassMLP(object):

    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=1, activation="relu", learning_rate=0.01):

        self.activation = activation
        self.learning_rate = learning_rate

        # input -> 1 or 2 hidden layers -> one output unit per class
        dims = [input_dim] + [hidden_dim] * num_layers + [num_classes]

        self.weights = []
        self.biases = []
        for i in range(len(dims) - 1):
            W = torch.randn(dims[i], dims[i + 1]) * (2 / dims[i]) ** .5
            b = torch.zeros(dims[i + 1])
            self.weights.append(W.requires_grad_())
            self.biases.append(b.requires_grad_())

    def activate(self, z):
        if self.activation == "relu":
            return torch.relu(z)
        elif self.activation == "tanh":
            return torch.tanh(z)

    def forward(self, X):
        a = X
        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            a = self.activate(torch.matmul(a, W) + b)
        out = torch.matmul(a, self.weights[-1]) + self.biases[-1]
        # Softmax over the class dimension so each row sums to 1
        return F.softmax(out, dim=-1)

    def cost(self, y_pred, y):
        # Categorical cross-entropy averaged over observations
        eps = torch.finfo(y_pred.dtype).eps
        y_pred = torch.clamp(y_pred, eps, 1 - eps)
        return -torch.mean(torch.sum(y * torch.log(y_pred), dim=-1))

    def fit(self, X, y, n_epochs=100):

        for epoch in range(n_epochs):

            epoch_cost = 0.0
            randperm = torch.randperm(X.shape[0])
            for ii in randperm:
                x_batch, y_batch = X[ii], y[ii]

                probs = self.forward(x_batch)
                cost = self.cost(probs, y_batch)
                cost.backward()
                epoch_cost += cost.item()

                with torch.no_grad():
                    for W, b in zip(self.weights, self.biases):
                        W -= self.learning_rate * W.grad
                        b -= self.learning_rate * b.grad

                    for W, b in zip(self.weights, self.biases):
                        W.grad.zero_()
                        b.grad.zero_()

            if epoch % 50 == 0:
                print(f"Epoch {epoch}, Average Cost: {epoch_cost / X.shape[0]:.4f}")
        return self

    def predict(self, X):
        with torch.no_grad():
            probs = self.forward(X)
            # Predicted class = unit with the highest probability
            predicted = torch.argmax(probs, dim=-1)
        return predicted

In [16]:
# Section 2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit the scaler on the training set only to avoid data leakage
scaler = StandardScaler()
X_train_scaled = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test_scaled  = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

# One-hot encode the labels for the cost; keep the integer labels for accuracy
encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = torch.tensor(encoder.fit_transform(y_train.reshape(-1, 1)), dtype=torch.float32)
y_test_onehot  = torch.tensor(encoder.transform(y_test.reshape(-1, 1)), dtype=torch.float32)

num_classes = y_train_onehot.shape[1]
print(X_train_scaled.shape, y_train_onehot.shape, num_classes)

torch.Size([800, 2]) torch.Size([800, 3]) 3


In [17]:
# Section 3
configs = [
    (1, 5, "relu"),
    (1, 10, "relu"),
    (1, 20, "relu"),
    (2, 10, "relu"),
    (1, 5, "tanh"),
    (1, 10, "tanh"),
    (1, 20, "tanh"),
    (2, 10, "tanh"),
]

mc_results = []
mc_models = {}

for layers, units, activation in configs:
    torch.manual_seed(1234)
    model = MultiClassMLP(input_dim=X_train_scaled.shape[1], hidden_dim=units, num_classes=num_classes,
                          num_layers=layers, activation=activation, learning_rate=0.01)
    model.fit(X_train_scaled, y_train_onehot, n_epochs=150)
    train_acc = accuracy_score(y_train, model.predict(X_train_scaled).numpy())
    holdout_acc = accuracy_score(y_test, model.predict(X_test_scaled).numpy())
    with torch.no_grad():
        train_cce = model.cost(model.forward(X_train_scaled), y_train_onehot).item()
        holdout_cce = model.cost(model.forward(X_test_scaled), y_test_onehot).item()
    mc_results.append({
        "Layers": layers,
        "Hidden Units": units,
        "Activation": activation,
        "Train Accuracy": train_acc,
        "Holdout Accuracy": holdout_acc,
        "Train CCE": train_cce,
        "Holdout CCE": holdout_cce,
    })
    mc_models[(layers, units, activation)] = model

mc_results_df = pd.DataFrame(mc_results)
mc_results_df.round(4)

Epoch 0, Average Cost: 0.9193
Epoch 50, Average Cost: 0.6158
Epoch 100, Average Cost: 0.6148
Epoch 0, Average Cost: 0.7171
Epoch 50, Average Cost: 0.5867
Epoch 100, Average Cost: 0.5744
Epoch 0, Average Cost: 0.7140
Epoch 50, Average Cost: 0.5761
Epoch 100, Average Cost: 0.5613
Epoch 0, Average Cost: 0.7616
Epoch 50, Average Cost: 0.5539
Epoch 100, Average Cost: 0.5385
Epoch 0, Average Cost: 0.8055
Epoch 50, Average Cost: 0.5720
Epoch 100, Average Cost: 0.5662
Epoch 0, Average Cost: 0.7216
Epoch 50, Average Cost: 0.5737
Epoch 100, Average Cost: 0.5557
Epoch 0, Average Cost: 0.6947
Epoch 50, Average Cost: 0.5641
Epoch 100, Average Cost: 0.5545
Epoch 0, Average Cost: 0.7543
Epoch 50, Average Cost: 0.5618
Epoch 100, Average Cost: 0.5439


,Layers,Hidden Units,Activation,Train Accuracy,Holdout Accuracy,Train CCE,Holdout CCE
0,1,5,relu,0.7912,0.845,0.6248,0.5494
1,1,10,relu,0.8100,0.855,0.5500,0.5393
2,1,20,relu,0.8125,0.850,0.5485,0.5425
3,2,10,relu,0.8025,0.800,0.5398,0.6099
4,1,5,tanh,0.7975,0.830,0.5612,0.5611
5,1,10,tanh,0.8100,0.850,0.5405,0.5548
6,1,20,tanh,0.8100,0.840,0.5460,0.5687
7,2,10,tanh,0.8038,0.815,0.5323,0.6377


Multi-class classification reflection and discussion

1. Increasing the number of hidden units from 5 to 10 improved performance for both activation functions. For ReLU, training accuracy rose from 79.12% to 81.00% and holdout accuracy from 84.5% to 85.5%, while training CCE dropped from 0.6248 to 0.5500 and holdout CCE from 0.5494 to 0.5393. For Tanh, training accuracy rose from 79.75% to 81.00% and holdout accuracy from 83.0% to 85.0%, and holdout CCE dropped from 0.5611 to 0.5548. Going from 10 to 20 units gave no further benefit. For ReLU, all metrics stayed nearly the same (holdout accuracy 85.0%, holdout CCE 0.5425). For Tanh, holdout accuracy fell from 85.0% to 84.0% and holdout CCE rose from 0.5548 to 0.5687, an early sign that the extra capacity started fitting noise. With only two features and three fairly separable clusters, about 10 hidden units is enough to capture the class boundaries.
2. Adding a second hidden layer (10 units in each) caused clear overfitting for both activation functions. Training CCE decreased (ReLU: 0.5500 to 0.5398, Tanh: 0.5405 to 0.5323), so the deeper models fit the training data more confidently. However, holdout CCE increased sharply (ReLU: 0.5393 to 0.6099, Tanh: 0.5548 to 0.6377), and holdout accuracy dropped (ReLU: 85.5% to 80.0%, Tanh: 85.0% to 81.5%). Training improving while holdout gets worse is the overfitting pattern we described in Section 1: the extra layer gives the model enough flexibility to bend its boundaries around the mislabeled observations, which does not generalize to unseen data. Since the data has this much label noise, the simpler one-layer model generalizes better.
3. ReLU and Tanh performed similarly overall, but showed the same tradeoff we saw in the binary task. Tanh reached a lower training CCE in every configuration (e.g., 0.5612 vs 0.6248 with 5 units), meaning it fit the training data more closely. ReLU, however, achieved a lower holdout CCE in every configuration and an equal or higher holdout accuracy, so it generalized better. ReLU's gap was biggest with only 5 units, likely because some ReLU units can become inactive ("die"), which leaves an already small layer with even less capacity. Tanh was also hit harder by the second layer (holdout CCE of 0.6377 vs 0.6099). The best overall model was the one-layer, 10-unit ReLU network, which had both the highest holdout accuracy (85.5%) and the lowest holdout CCE (0.5393).

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [37]:
# importing packages
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [38]:
# importing data
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [39]:
# implementation
class RegressionMLP(object):
    
    def __init__(self, 
                 input_dim=2, 
                 hidden_dim=10, 
                 num_layers=1,
                 activation='relu'):

        self.activation = activation

        dims = [input_dim] + [hidden_dim] * num_layers + [1]

        self.weights = []
        self.biases = []

        for i in range(len(dims) - 1):
            W = torch.randn(dims[i], dims[i+1]) * np.sqrt(2 / dims[i])
            b = torch.zeros(1, dims[i+1])

            self.weights.append(W.requires_grad_())
            self.biases.append(b.requires_grad_())
    
    def forward(self, X):
        a = X 

        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            z = a @ W + b
        
            if self.activation == "relu":
                a = torch.relu(z)
            elif self.activation == "tanh":
                a = torch.tanh(z)

        output = a @ self.weights[-1] + self.biases[-1]
        return output

    def cost(self, y_pred, y_true):
        return torch.mean((y_pred - y_true) ** 2)
    
    def fit(self, X, y, lr=0.01, epochs=150, X_holdout=None, y_holdout=None):
        self.loss_history = []
        self.holdout_loss_history = []

        for epoch in range(epochs):
            indices = torch.randperm(len(X))

            for i in indices:
                xi = X[i:i+1]
                yi = y[i:i+1]

                pred = self.forward(xi)
                loss = self.cost(pred, yi)

                loss.backward()

                with torch.no_grad():
                    for W, b in zip(self.weights, self.biases):
                        W -= lr * W.grad
                        b -= lr * b.grad

                        W.grad.zero_()
                        b.grad.zero_()

            with torch.no_grad():
                epoch_loss = self.cost(self.forward(X), y).item()
                self.loss_history.append(epoch_loss)

                if X_holdout is not None and y_holdout is not None:
                    holdout_loss = self.cost(self.forward(X_holdout), y_holdout).item()
                    self.holdout_loss_history.append(holdout_loss)

        return self
        
    def predict(self, X):
        with torch.no_grad():
            return self.forward(X)


In [40]:
# data preparation (train/test split, standardization, PyTorch conversion)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, random_state=1234
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_holdout = scaler.transform(X_holdout)

y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_holdout = y_scaler.transform(y_holdout.reshape(-1, 1))

X_train = torch.tensor(X_train, dtype=torch.float32)
X_holdout = torch.tensor(X_holdout, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_holdout = torch.tensor(y_holdout, dtype=torch.float32)

In [41]:
# training and evaluation

configs = [
    (1, 5, "relu"),
    (1, 10, "relu"),
    (2, 10, "relu"),
    (1, 5, "tanh"),
    (1, 10, "tanh"),
    (2, 10, "tanh")]

results = []
models = {}

for layers, units, activation in configs:
    torch.manual_seed(1234)

    model = RegressionMLP(
        input_dim=2,
        hidden_dim=units,
        num_layers=layers,
        activation=activation
    )

    model.fit(X_train, y_train, lr=0.01, epochs=150, X_holdout=X_holdout, y_holdout=y_holdout)

    train_pred = y_scaler.inverse_transform(model.predict(X_train).numpy())
    holdout_pred = y_scaler.inverse_transform(model.predict(X_holdout).numpy())

    train_mse = mean_squared_error(y_scaler.inverse_transform(y_train.numpy()), train_pred)
    holdout_mse = mean_squared_error(y_scaler.inverse_transform(y_holdout.numpy()), holdout_pred)
    
    results.append({
        "Layers": layers,
        "Hidden Units": units,
        "Activation": activation,
        "Train MSE": train_mse,
        "Holdout MSE": holdout_mse
    })

    models[(layers, units, activation)] = model

results_df = pd.DataFrame(results)
results_df.round(2)

,Layers,Hidden Units,Activation,Train MSE,Holdout MSE
0,1,5,relu,5535.83,5502.25
1,1,10,relu,5529.49,5637.13
2,2,10,relu,5449.96,5561.79
3,1,5,tanh,5515.64,5266.51
4,1,10,tanh,5722.61,5706.75
5,2,10,tanh,5434.20,5738.90


Regression reflection and discussion

1. How does the number of hidden units affect performance?
- Increasing the number of hidden units from 5 to 10 did not consistently improve performance. For ReLU, training MSE decreased slightly from 5535.83 to 5529.49, while holdout MSE increased from 5502.25 to 5637.13. For Tanh, both training and holdout MSE increased, suggesting that additional hidden units did not improve the model's ability to learn or generalize. Since the dataset has a relatively simple underlying linear relationship, increasing model complexity may not provide substantial benefits.
2. What changes when using two layers instead of one?
- Adding a second hidden layer improved training performance for both activation functions when compared to the corresponding one-layer, 10-unit models. ReLU training MSE decreased from 5529.49 to 5449.96, while Tanh training MSE decreased from 5722.61 to 5434.20. However, holdout performance improved slightly for ReLU but worsened for Tanh. This suggests that deeper networks can fit the training data more effectively, but increased complexity does not necessarily improve generalization.
3. How does the activation function (ReLU vs. Tanh) influence results?
- The impact of activation function varied across architectures. Overall, the one-layer Tanh model with 5 hidden units achieved the lowest holdout MSE, suggesting that a relatively simple architecture was sufficient for this dataset. However, ReLU performed better with 1 hidden layer and 10 units, as well as with 2 hidden layers and 10 units. This suggests that neither activation function consistently outperformed the other. Tanh can be effective at modeling smooth relationships, while ReLU may train more efficiently because it avoids saturation for positive inputs. However, because neural networks are sensitive to random initialization and optimization, these results do not necessarily imply that this architecture will consistently outperform the others across repeated experiments.

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

Because each of us implemented a separate class for our task, much of our code ended up being duplicated. Comparing the three classes that we created, there are nearly identical pieces in all of them. All three build the network in the `__init__` method the same way, by creating a list of layer sizes and looping over it to initialize weights and biases. All of them select hidden layer activations (ReLU, Tanh) with the same conditional logic, and all three pass inputs through the hidden layers with the same loop in `forward`. Most importantly, they all use the exact same stochastic gradient descent loop in `fit`: shuffling the training data with `torch.randperm`, running a forward pass on each sample, computing the cost, calling `backward()` to compute the gradients, and then updating every weight and bias and zeroing the gradients. The parts that actually differ between the three tasks are small. Binary classification uses one output unit with a sigmoid activation function, binary cross-entropy as the cost, and predicts by thresholding the probability at 0.5. Multi-class classification uses one output per class with a softmax activation, categorical cross-entropy as the cost, and predicts the class with the highest probability using `argmax`. Regression uses one output unit with no activation, mean squared error as the cost, and returns the raw output as its prediction.
We could have used inheritance to separate these shared and task-specific parts. A parent class, `BaseMLP`, could contain everything the three models have in common, including:
- the `__init__` method, with an added output dimension argument for the number of output units
- hidden layer activations
- hidden layer loop in `forward`
- the entire `fit` method

The final step of `forward` would call a method named `output_activation`, and `fit` would call `self.cost`, without either method needing to know which task it is solving. Each task would be a small subclass that overrides only the three methods that differ: `output_activation`, `cost`, and `predict`. For example:
- `BinaryMLP` would define the output activation as `torch.sigmoid`, cost as binary cross-entropy, and `predict` as probability > 0.5.
- `MultiClassMLP` would use softmax, categorical cross-entropy, and `argmax`.
- `RegressionMLP` would return its input unchanged in `output_activation`, use mean squared error, and return the raw outputs as `predict`.

This would use the same approach as the tutorial, where `LogisticRegressionSGD` inherits from `LinearRegressionSGD` and only overrides its `forward` and `loss` methods. By using inheritance in our code, we could have made our code more reusable and easier to maintain. Any changes that we wanted to make to the base methods could have been done within the `BaseMLP` class and would then automatically apply to all the inheriting models. Any changes that we needed to make to the specific models themselves can be implemented within their classes. 

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

1. During this lab, because all 3 coding parts were entirely separate, our group did not communicate as frequently as the past labs. We all worked together during the lab period, but upon leaving the room, we divided up the work and communicated solely via text. Since no one needed the previous member's part to do theirs, we were communicating less frequently with one another. 
2. In the future, we will work more collaboratively outside of the classroom by setting up in person times to meet. This will allow us to be more on top of deadlines and understand better how each group member thinks and works. 
